# StockSense forecasting research: experiments, comparison and uncertainty

This notebook documents the research progression and provides **opt-in, offline** comparison code. It is separate from the authenticated application's saved-XGBoost CSV/JSON inference. Running the default cells reads existing data and reports; it does not download data, fit models, modify prepared datasets or write artifacts.

The objective is a decimal **10-trading-session adjusted-close return**, with an empirical prediction interval. Forecasts are available after the origin session closes. The frozen pretrained **Chronos-2 Transformer was evaluated zero-shot**; StockSense did not train or fine-tune its weights.

This notebook contains the executable tabular training path and recorded outputs from completed runs. The separate Chronos-2 notebook contains the executable pretrained-model benchmark. Historical tables are clearly labeled when they are recorded rather than recomputed in the current kernel. Different snapshots, feature contracts, folds and refit policies must not be combined into a single leaderboard.


## Research progression and evidence

| Stage | Inputs / question | Recorded finding |
|---|---|---|
| Original price-only comparison | Prepared 34-predictor allowlist; zero, Ridge, Random Forest, XGBoost | Validation RMSE: zero 7.1688 pp, Ridge 9.8840, Random Forest 8.1794, XGBoost 7.6646. Learned models initially lost. |
| Early stopping | Stop boosting using development outcomes | One-round XGBoost RMSE 7.1073 pp; training-mean baseline 7.1033 pp. Improvement over zero did not establish feature skill. |
| Relative features | 27 predictors; normalize absolute price indicators | Relative Ridge RMSE 7.1330 pp versus original Ridge 9.8840; still worse than the training-mean baseline. |
| Expanded features | 43 predictors including longer momentum, volatility, money flow and relative strength | Seven-round XGBoost validation RMSE 7.0998 pp; about 0.05% improvement versus mean. Bootstrap intervals included zero. |
| Walk-forward / rolling history | Purged 2023/2024 development evaluations | Small 2024 gains did not repeat reliably in 2023. |
| PCA and compact predictors | Inner-selected PCA components; simplified features | PCA did not consistently beat constants; compact Ridge/XGBoost also lacked consistent improvement. |
| Longer history | Colab Yahoo snapshot, 2010–2024; corrected RSI and volume changes | Seven retrospective years, 2018–2024, provided more market regimes. This longer snapshot is **not** the committed short-history dataset. |
| Ensembles | Fixed 50/50 Ridge–XGBoost and baseline-anchored blend | Small, inconsistent gains; intervals generally included zero. |
| SEC pilot | Acceptance-dated revenue/net income, AAPL/MSFT/NVDA; margin/age features | Matched fundamentals did not consistently help the three-company sample. No universal claim about fundamentals. |
| Additional families | Elastic Net, Extra Trees, Histogram Gradient Boosting | Ridge retained lowest average error in that experiment; all 28 displayed annual 20-session MAE gain intervals crossed zero. |
| Tabular accuracy | Regression predictions converted to UP / NOT UP | Mean/median 56.72% vs XGBoost 56.69%; balanced accuracy about 50%. No directional advantage demonstrated. |
| Repository XGBoost calibration | Explicit XGBoost family; bounded inner grid; refit before separate calibration | Snapshot inference implemented; nominal 90% intervals undercovered in partial 2026. No claim of overall best family or live validated forecasting. |
| Chronos-2 zero-shot Transformer | Frozen `amazon/chronos-2`; 252-session price context; 10-session horizon; no fine-tuning | Full 2024–2026 evaluation did not beat zero return on MAE/RMSE; balanced accuracy stayed near 50%. Native interval coverage was about 87–88%. |
| Point-in-time SEC comparison | Acceptance-dated ratios joined backward to price origins; identical XGBoost folds | Price + SEC was effectively identical to price-only in 2024/2025 and changed 2026 MAE by only 0.0003 pp; block intervals included zero. |

Earlier experiment notebooks  included `StockSense_Forecasting_Experiment.ipynb`, `StockSense_Additional_Models_Experiment (1).ipynb`, and `StockSense_Tabular_Models_Accuracy (1).ipynb`. Their saved outputs/source were inspected when compiling this notebook.


## Historical additional-model leaderboard

Equal-year average errors for the longer-history 2018–2024 experiment, **historically reported, not recomputed here**:

| Family | MAE (pp) | RMSE (pp) |
|---|---:|---:|
| Ridge | 4.677412 | 6.586109 |
| Histogram Gradient Boosting | 4.683204 | 6.590801 |
| Extra Trees | 4.685253 | 6.597498 |
| Elastic Net | 4.689765 | 6.592175 |
| Training median | 4.692237 | 6.598296 |
| Training mean | 4.693540 | 6.600099 |
| Zero return | 4.746255 | 6.649570 |

Ridge's average MAE gain over median was approximately 0.32%; that alone did not prove a reliable advantage. The consolidated comparison trains on the current 27-feature contract. Historical 34-feature and expanded 43-feature results remain recorded separately because combining feature snapshots would invalidate the comparison. SEC has its own point-in-time comparison, and Chronos-2 was evaluated zero-shot rather than trained.


## Setup: discover the checkout and use existing dependencies

Start Jupyter from this checkout, or open the notebook within a checkout. Install the documented offline forecasting requirements in your own environment beforehand if needed. No installation/download cell runs here. `RUN_ALL_MODEL_TRAINING=False` is the default; the later comparison can take substantial CPU time when enabled.


In [ ]:
from pathlib import Path
import sys, json, hashlib, importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/forecasting/inputs.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the StockSense checkout.")
sys.path.insert(0, str(ROOT))
RUN_ALL_MODEL_TRAINING = False          # Broad multi-family comparison below.
RUN_REPOSITORY_TRAINING = False  # Exact bounded XGBoost/Ridge/baseline walk-forward.
SAVE_ARTIFACTS = False
AS_OF = "2026-09-18"  # Fixed historical snapshot cutoff; not today's prices.
EVALUATION_YEARS = (2024, 2025, 2026)
from data.forecasting.inputs import build_inputs, RELATIVE_FEATURES
from data.forecasting.sources import load_local
from data.prepare_market_data import PRICE_FEATURES
from data.forecasting.models import _quantile, _scale
print({"all_model_training": RUN_ALL_MODEL_TRAINING,
       "repository_training": RUN_REPOSITORY_TRAINING,
       "save_artifacts": SAVE_ARTIFACTS})
print({name: importlib.metadata.version(name)
       for name in ("numpy", "pandas", "scikit-learn", "joblib")})


## Load and fingerprint the existing numerical source

Prepared candidates normally live under `data/artifacts/market/price_only_features.parquet`. When absent, use the audited committed `forecast_features.parquet`, whose allowed numerical inputs are equivalent. Raw prices validate prices/targets and provide past-only corrective calculations. Do not silently substitute macro-enriched data.

The historical committed prepared snapshot was 110,229 rows, 52 columns, 87 tickers, dated 2021-09-01–2026-09-18; its raw counterpart had 135,593 rows. Loading a different snapshot can change these values.


In [ ]:
candidate = ROOT / "data/artifacts/market/price_only_features.parquet"
feature_path = candidate if candidate.is_file() else ROOT / "data/processed/forecast_features.parquet"
price_path = ROOT / "data/raw/historical_prices.parquet"
universe_path = ROOT / "data/asset_universe.csv"
features, prices, universe, provenance = load_local(feature_path, price_path, universe_path)
# Relative paths in display; provenance retains hashes for reproducibility.
display(pd.DataFrame([
    {"dataset": name, "file": Path(info["path"]).relative_to(ROOT).as_posix(),
     "rows": info["rows"], "columns": len(info["columns"]), "sha256": info["sha256"]}
    for name, info in provenance["datasets"].items()
]))
frame = build_inputs(features, prices, universe, as_of=AS_OF)
print("Validated relative dataset:", frame.shape)
print("Origin dates:", frame.Date.min(), frame.Date.max())
print("Tickers:", frame.ticker.nunique(), "known targets:", frame.target_return_10d.notna().sum())
display(frame.head())


## Feature and target contracts

`PRICE_FEATURES` is the original **34-column preparation allowlist**, not the experimental relative list. `RELATIVE_FEATURES` is the **27-column forecasting input contract**. Identifiers, target, outcome date and training flag stay outside X.

For ticker-specific ordered prices: `y[t] = adjusted_close[t+10] / adjusted_close[t] - 1`. Backward `return_10d` instead uses `price[t] / price[t-10] - 1`. Relative moving-average features use `price / moving_average - 1`; MACD/ATR are divided by origin price. RSI is corrected to 100 for gains with zero losses and 50 for no movement. Volume change after previous zero volume is undefined and remains missing for training-only imputation.

Same-session SPY/VIX and price indicators require finalized after-close observations. Revisions to adjusted historical prices and the fixed current universe remain provenance limitations.


In [ ]:
print("Original prepared allowlist:", len(PRICE_FEATURES))
print("\n".join(PRICE_FEATURES))
print("\nExperimental relative predictors:", len(RELATIVE_FEATURES))
print("\n".join(RELATIVE_FEATURES))
assert len(RELATIVE_FEATURES) == 27 and len(set(RELATIVE_FEATURES)) == 27
assert not {"Date", "ticker", "target_return_10d", "label_end_date", "is_training_row"}.intersection(RELATIVE_FEATURES)
assert not np.isinf(frame[list(RELATIVE_FEATURES)].to_numpy()).any()
display(frame[list(RELATIVE_FEATURES)].isna().sum().loc[lambda s: s > 0].to_frame("missing"))


## Purged yearly folds

Training precedes July of the previous year; July–September selects parameters; October–December calibrates uncertainty; the following year evaluates. Every earlier row's outcome must mature **strictly before** the next boundary. Current short-history inputs cannot support the 2023 fold with 252 minimum training observations: only 199 purged observations per ticker exist. Consequently the example uses 2024, 2025 and partial 2026.

All these years have already been inspected. They are retrospective development diagnostics, not fresh held-out confirmation. The longer-history 2018–2024 experiments are distinct and require their original Yahoo snapshot.


In [ ]:
TARGET = "target_return_10d"

def make_fold(data, year):
    tuning = pd.Timestamp(year - 1, 7, 1)
    calibration = pd.Timestamp(year - 1, 10, 1)
    evaluation = pd.Timestamp(year, 1, 1)
    end = pd.Timestamp(year + 1, 1, 1)
    known = data[TARGET].notna()
    masks = {
        "training": known & data.Date.lt(tuning) & data.label_end_date.lt(tuning),
        "inner": known & data.Date.ge(tuning) & data.Date.lt(calibration) & data.label_end_date.lt(calibration),
        "calibration": known & data.Date.ge(calibration) & data.Date.lt(evaluation) & data.label_end_date.lt(evaluation),
        "evaluation": known & data.Date.ge(evaluation) & data.Date.lt(end) & data.label_end_date.lt(end),
    }
    counts = data.loc[masks["training"]].groupby("ticker").size()
    eligible = counts[counts >= 252].index
    parts = {name: data.loc[mask & data.ticker.isin(eligible)].sort_values(["Date", "ticker"])
             for name, mask in masks.items()}
    if any(part.empty for part in parts.values()):
        raise ValueError(f"Insufficient rows in fold {year}")
    for name, boundary in (("training", tuning), ("inner", calibration), ("calibration", evaluation), ("evaluation", end)):
        assert parts[name].label_end_date.lt(boundary).all()
    return parts

folds = {year: make_fold(frame, year) for year in EVALUATION_YEARS}
display(pd.DataFrame([
    {"year": year, "partition": name, "rows": len(part), "dates": part.Date.nunique(),
     "first_origin": part.Date.min(), "last_origin": part.Date.max(),
     "latest_outcome": part.label_end_date.max()}
    for year, parts in folds.items() for name, part in parts.items()
]))


## Full bounded model definitions and training pipelines

These are executable training pipelines for every candidate family in the consolidated comparison. Earlier one-off notebooks sometimes used different bounded grids, so their historical scores remain separately labeled rather than silently combined. Pipelines fit imputation/scaling only on each training partition. PCA component count is selected on the inner window. A fixed equal-weight blend is defined in advance. The comparison does **not** refit on inner labels; the saved repository walk-forward experiment below does refit before calibration. That distinction can change results.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor, VotingRegressor
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, balanced_accuracy_score

FEATURES = list(RELATIVE_FEATURES)

class TrainingConstantRegressor(BaseEstimator, RegressorMixin):
    """Transparent zero/mean/median baseline fitted only on training labels."""

    def __init__(self, strategy="mean", constant=0.0):
        self.strategy = strategy
        self.constant = constant

    def fit(self, X, y):
        values = np.asarray(y, dtype=float)
        if not np.isfinite(values).all() or len(values) == 0:
            raise ValueError("Baseline training labels must be finite and nonempty")
        if self.strategy == "zero":
            self.value_ = float(self.constant)
        elif self.strategy == "mean":
            self.value_ = float(np.mean(values))
        elif self.strategy == "median":
            self.value_ = float(np.median(values))
        else:
            raise ValueError("strategy must be zero, mean or median")
        return self

    def predict(self, X):
        if not hasattr(self, "value_"):
            raise RuntimeError("Fit the baseline before prediction")
        return np.full(len(X), self.value_, dtype=float)

def linear(estimator, components=None):
    steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    if components is not None:
        steps.append(("pca", PCA(n_components=components, svd_solver="full")))
    return Pipeline(steps + [("model", estimator)])

def trees(estimator):
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", estimator)])

candidates = {
    "Zero": [TrainingConstantRegressor(strategy="zero", constant=0.0)],
    "Mean": [TrainingConstantRegressor(strategy="mean")],
    "Median": [TrainingConstantRegressor(strategy="median")],
    "Ridge": [linear(Ridge(alpha=a)) for a in (100., 1000.)],
    "Elastic Net": [linear(ElasticNet(alpha=a, l1_ratio=.5, max_iter=10000)) for a in (.0001, .001, .01)],
    "Random Forest": [trees(RandomForestRegressor(n_estimators=120, max_depth=d, min_samples_leaf=100,
                            max_features=.7, n_jobs=2, random_state=42)) for d in (6, 12)],
    "Extra Trees": [trees(ExtraTreesRegressor(n_estimators=120, max_depth=d, min_samples_leaf=100,
                          max_features=.7, n_jobs=2, random_state=42)) for d in (6, 12)],
    "Histogram GB": [trees(HistGradientBoostingRegressor(max_iter=n, learning_rate=.03, max_leaf_nodes=7,
                          min_samples_leaf=100, l2_regularization=10, early_stopping=False, random_state=42)) for n in (30, 100)],
    "PCA Ridge": [linear(Ridge(alpha=100), components=k) for k in (5, 10, 20)],
}
try:
    from xgboost import XGBRegressor
    candidates["XGBoost"] = [trees(XGBRegressor(n_estimators=n, max_depth=2, learning_rate=.03,
        min_child_weight=100, reg_lambda=20, objective="reg:squarederror", tree_method="hist",
        n_jobs=2, random_state=42)) for n in (1, 10, 50)]
    candidates["50/50 ensemble"] = [VotingRegressor([
        ("ridge", linear(Ridge(alpha=100))), ("xgb", clone(candidates["XGBoost"][1]))], weights=[.5, .5])]
except ImportError:
    print("XGBoost is not installed: its family and blend are unavailable in this example.")
print({family: len(options) for family, options in candidates.items()})
print("Definitions only: nothing fitted.")


## Training-code inventory in this notebook

The preceding code cell contains the actual scikit-learn/XGBoost estimator and preprocessing definitions used for the tabular comparisons. The following opt-in comparison cell fits and evaluates them on identical purged folds.

| Candidate | Implementation included here | Learned model? |
|---|---|---|
| Zero-return baseline | `TrainingConstantRegressor(strategy="zero")` | No |
| Training-mean baseline | `TrainingConstantRegressor(strategy="mean")` | No |
| Training-median baseline | `TrainingConstantRegressor(strategy="median")` | No |
| Ridge | Median imputation → scaling → `Ridge` | Yes |
| Elastic Net | Median imputation → scaling → `ElasticNet` | Yes |
| Random Forest | Median imputation → `RandomForestRegressor` | Yes |
| Extra Trees | Median imputation → `ExtraTreesRegressor` | Yes |
| Histogram Gradient Boosting | Median imputation → `HistGradientBoostingRegressor` | Yes |
| PCA Ridge | Median imputation → scaling → PCA → Ridge | Yes |
| XGBoost | Median imputation → bounded `XGBRegressor` candidates | Yes |
| Ridge–XGBoost ensemble | Fixed 50/50 `VotingRegressor` | Yes |

The later “Exact repository training path” cell contains the actual production-candidate walk-forward call with the repository's 12-configuration XGBoost grid, refitting and separate uncertainty calibration. Chronos-2 is not listed as a trained model because StockSense did not train it; its complete frozen zero-shot inference code remains in the linked Chronos notebook.


## Train, select, calibrate and evaluate every candidate family

Set `RUN_ALL_MODEL_TRAINING=True` deliberately to execute. Each family picks its parameter using inner MAE (RMSE tie-break). Calibration residuals are computed only afterwards; outer outcomes never select a parameter. Every family gets its own uncertainty quantile and uses identical evaluation rows.

`scale = max(annualized_volatility_20 × sqrt(10/252), 0.005)`; `score = abs(error)/scale`; `q` is the finite-sample upper rank at nominal 90%; interval is prediction ± q × scale. This is empirical calibration on dependent, overlapping stock returns: **no IID coverage guarantee**. Width in percentage points is an uncertainty measurement, not confidence or probability of correctness.


In [ ]:
experiment_rows, selected_estimators, predictions = [], {}, {}
if not RUN_ALL_MODEL_TRAINING:
    print("Skipped fitting. Enable RUN_ALL_MODEL_TRAINING only for an intentional retrospective rerun.")
else:
    for year, parts in folds.items():
        tr, inner, cal, ev = (parts[name] for name in ("training", "inner", "calibration", "evaluation"))
        majority_up = tr[TARGET].gt(0).mean() > .5
        for family, options in candidates.items():
            scored = []
            for index, option in enumerate(options):
                fitted = clone(option).fit(tr[FEATURES], tr[TARGET])
                p_inner = fitted.predict(inner[FEATURES])
                scored.append((mean_absolute_error(inner[TARGET], p_inner),
                               np.sqrt(mean_squared_error(inner[TARGET], p_inner)), index, fitted))
            inner_mae, inner_rmse, selected_index, fitted = min(scored, key=lambda value: value[:3])
            residuals = np.abs(cal[TARGET].to_numpy() - fitted.predict(cal[FEATURES]))
            q = _quantile(residuals / _scale(cal), .9)
            p = fitted.predict(ev[FEATURES]); y = ev[TARGET].to_numpy()
            half = q * _scale(ev); lower, upper = p-half, p+half
            interval_score = upper-lower + 20*((lower-y)*(y<lower)+(y-upper)*(y>upper))
            experiment_rows.append({"year": year, "model": family, "candidate_index": selected_index,
                "MAE_pp": mean_absolute_error(y,p)*100, "RMSE_pp": np.sqrt(mean_squared_error(y,p))*100,
                "R2": r2_score(y,p), "direction_accuracy_pct": np.mean((y>0)==(p>0))*100,
                "balanced_accuracy_pct": balanced_accuracy_score(y>0,p>0)*100,
                "always_up_accuracy_pct": np.mean(y>0)*100,
                "training_majority_accuracy_pct": np.mean((y>0)==majority_up)*100,
                "interval_coverage": np.mean((y>=lower)&(y<=upper)),
                "width_pp": np.mean(upper-lower)*100, "interval_score_pp": interval_score.mean()*100,
                "inner_MAE_pp": inner_mae*100, "calibration_quantile": q})
            selected_estimators[(year,family)] = fitted
            predictions[(year,family)] = ev[["Date","ticker",TARGET]].assign(prediction=p, lower=lower, upper=upper)
        print(year, "completed")
results = pd.DataFrame(experiment_rows)
if not results.empty:
    display(results.round(4))
    display(results.groupby("model")[["MAE_pp","RMSE_pp","direction_accuracy_pct","balanced_accuracy_pct",
                                      "interval_coverage","width_pp"]].mean().sort_values("MAE_pp").round(4))


## Reading accuracy and selecting a research candidate

Direction is positive return = UP; zero or negative = NOT UP. Accuracy near 56% is not evidence of skill if always-UP scores 56–58%. Balanced accuracy near 50% indicates little discrimination. Return-model selection uses MAE, with RMSE secondary; direction is a separate diagnostic. A baseline can win.

Outer-year tables here are already-inspected development evidence. Choosing a family after examining them requires a new prospective evaluation before any independent superiority claim. Do not retune based on calibration coverage or the final year's score and then call that same year a test.


## Read the actual repository experiment, without unpickling

This cell reads optional `report.json` only. Ignored run artifacts are not expected in a fresh Git clone. If absent, use the static documented historical table below; do not manufacture fresh outputs or silently train. The actual module uses the predeclared 12-configuration XGBoost grid, selects on inner MAE, then refits on labels matured before calibration. XGBoost is explicitly requested; the run does **not** prove it is the overall best family.


In [ ]:
report_path = ROOT / "data/artifacts/forecasting/walkforward-xgboost-2026-09-18/report.json"
repository_report = None
if report_path.is_file():
    repository_report = json.loads(report_path.read_text())
    actual_rows = []
    for fold in repository_report["walk_forward"]:
        metrics = fold["xgboost"]
        actual_rows.append({"year": fold["year"], "selected": fold["metadata"]["selected_model"],
            "MAE_pp": metrics["MAE_pp"], "median_MAE_pp": fold["baselines"]["median"]["MAE_pp"],
            "accuracy_pct": metrics["direction_accuracy_pct"], "always_up_pct": metrics["always_up_accuracy_pct"],
            "coverage": metrics["interval_coverage"], "width_pp": metrics["mean_interval_width_pp"]})
    display(pd.DataFrame(actual_rows).round(4))
    display(pd.DataFrame([{"year": fold["year"], **interval}
        for fold in repository_report["walk_forward"] for interval in fold["paired_mae_intervals"]]))
else:
    print("Ignored repository run report is unavailable. Historical recorded values are documented below.")


### Actual recorded repository results — not recomputed here

| Year | XGBoost MAE (pp) | Median MAE (pp) | Direction / always-UP (%) | Interval coverage | Mean width (pp) |
|---|---:|---:|---:|---:|---:|
| 2024 | 4.685606 | 4.691181 | 57.7230 / 57.7230 | 91.9445% | 23.7303 |
| 2025 | 5.103544 | 5.119377 | 56.6140 / 56.6140 | 92.9262% | 27.4357 |
| 2026, partial | 5.788989 | 5.796326 | 52.6695 / 52.6695 | 86.9278% | 23.9618 |

Selected depth-1 learning-rate .03 models used 10, 10 and 150 boosting rounds respectively. All three had balanced accuracy 50%. MAE gains against median were 0.005575, 0.015833 and 0.007337 pp. The 2024 paired bootstrap intervals excluded zero; 2025/2026 crossed zero for all 10/20/40-session blocks. Repeated experimentation and no multiple-testing correction limit interpretation. Partial 2026 **undercovered** relative to nominal 90%; do not promise calibrated 90% future coverage.


## Exact repository training path (actual XGBoost code)

This is the same implementation used to create the saved application candidate. It calls `data.forecasting.models.train_model` with the repository's predeclared 12-configuration XGBoost grid. For each year it:

1. trains only on earlier matured labels;
2. chooses hyperparameters on July–September inner MAE (RMSE tie-breaker);
3. refits the chosen configuration using labels matured before October;
4. calibrates a frozen model on October–December residuals; and
5. evaluates the following year without fitting on that year's outcomes.

Set `RUN_REPOSITORY_TRAINING=True` to perform the real CPU training. The cell does not download data. Artifact writing remains separately gated.


In [ ]:
from data.forecasting.models import train_model, evaluate
from data.forecasting.walkforward import GRID, paired_intervals

def train_repository_fold(data: pd.DataFrame, year: int):
    prior = year - 1
    development = data.loc[data.Date.lt(f"{year + 1}-01-01")].copy()
    bundle = train_model(
        development,
        tuning_start=f"{prior}-07-01",
        calibration_start=f"{prior}-10-01",
        evaluation_start=f"{year}-01-01",
        model="xgboost",
        xgboost_grid=GRID,
        refit_selected=True,
    )
    evaluation = development.loc[
        development.Date.ge(f"{year}-01-01")
        & development[TARGET].notna()
        & development.label_end_date.lt(f"{year + 1}-01-01")
        & development.ticker.isin(bundle.metadata["eligible_tickers"])
    ].copy()
    return bundle, evaluate(bundle, evaluation), paired_intervals(bundle, evaluation)

actual_training_rows = []
actual_trained_bundles = {}
if RUN_REPOSITORY_TRAINING:
    for year in EVALUATION_YEARS:
        bundle, metrics, intervals = train_repository_fold(frame, year)
        actual_trained_bundles[year] = bundle
        actual_training_rows.append({
            "year": year,
            "selected_model": bundle.metadata["selected_model"],
            "MAE_pp": metrics["MAE_pp"],
            "RMSE_pp": metrics["RMSE_pp"],
            "direction_accuracy_pct": metrics["direction_accuracy_pct"],
            "balanced_accuracy_pct": metrics["balanced_accuracy_pct"],
            "interval_coverage": metrics["interval_coverage"],
            "mean_interval_width_pp": metrics["mean_interval_width_pp"],
        })
    display(pd.DataFrame(actual_training_rows).round(4))
else:
    print("Exact training code loaded but not executed. Set RUN_REPOSITORY_TRAINING=True to train all three yearly folds.")


## Why XGBoost was selected as the best overall deployable candidate

XGBoost was selected as the best overall **deployable learned-model candidate** after considering chronological error results, consistency, uncertainty integration and production cost—not because “56% accuracy” proved predictive skill:

- The bounded walk-forward XGBoost beat the training median on MAE in 2024, 2025 and partial 2026, although most block-bootstrap intervals still crossed zero.
- It beat Ridge on MAE in 2024 and 2025; Ridge was slightly better in partial 2026. No family dominated every period.
- It accepts nonlinear interactions among momentum, volatility, volume and market-context inputs without scaling.
- Median imputation, fixed seeds, a bounded grid and small trees make the offline run reproducible and inexpensive.
- It fits the existing saved batch contract: one compact model generates `forecasts.csv`, while calibration remains a separate auditable step.
- Chronos-2 was materially heavier and did not improve MAE/RMSE or balanced direction accuracy in the completed zero-shot benchmark.

Overall decision: XGBoost offered the strongest combined balance of learned nonlinear behavior, competitive walk-forward MAE, reproducibility, compact CPU inference and compatibility with calibrated uncertainty. It was therefore the best candidate for this application deployment. It was not the lowest-error method in every individual year, and the statistical evidence does not establish universal forecasting superiority. The application consequently labels its outputs as historical/experimental and exposes the wide uncertainty intervals.


### Overall selection matrix

| Criterion | XGBoost finding | Selection implication |
|---|---|---|
| Walk-forward MAE | Beat the median baseline in all three recorded years; beat Ridge in 2024/2025 | Competitive learned-model error |
| Stability | Gains were small and most confidence intervals crossed zero | Keep experimental labels and baselines |
| Direction | Raw accuracy resembled always-UP; balanced accuracy was 50% | Do not select or market it on “accuracy” |
| Uncertainty | Supports separate empirical calibration and per-ticker interval widths | Required for cautious product output |
| Runtime | Small bounded trees; offline CPU batch inference | Practical monthly refresh |
| Deployment | Existing validated CSV/report/database contract | Lowest integration risk |
| Chronos comparison | Chronos-2 had worse MAE/RMSE than zero return in every evaluated year | No reason to deploy the heavier Transformer |
| SEC comparison | SEC predictors did not measurably improve point forecasting | Keep SEC in the reasoning/explanation layer |

“Best overall” therefore means the best **deployment choice among the evaluated learned candidates for StockSense's current architecture**. It does not mean proven market-beating performance.


## Why another model was not selected

No evaluated learned model produced sufficiently consistent evidence to replace XGBoost:

- **Ridge** beat XGBoost in partial 2026 but was worse in 2024 and 2025.
- **Random Forest, Extra Trees, Elastic Net, Histogram Gradient Boosting and PCA Ridge** did not provide consistent, statistically supported gains across time.
- **Chronos-2** had worse MAE and RMSE than the zero-return baseline in every evaluated year, despite sometimes showing higher raw directional accuracy.
- **Price + SEC XGBoost** was effectively identical to price-only XGBoost; the small 2026 difference was not statistically established.
- **Ridge–XGBoost and baseline-anchored ensembles** produced small, inconsistent changes whose confidence intervals generally crossed zero.
- **Zero, training-mean and training-median baselines** remained very competitive, demonstrating that the available price features contain limited stable ten-session signal.

If selection were based only on minimizing retrospective error, a constant baseline would often be the safest choice. It would, however, provide the same forecast across many assets and would not demonstrate learned ticker-specific variation. XGBoost was retained for the research application because it produces asset-specific estimates, remains operationally small and reproducible, and supports the existing uncertainty-calibration and deployment contracts.

> **Model-selection conclusion:** XGBoost was selected as the best overall deployable learned model, although it did not demonstrate strong or statistically consistent superiority over simple baselines. StockSense manages this limitation through empirical uncertainty intervals, conservative rule-engine gating, stale-data labels and explicit experimental disclosure.

A Transformer, LSTM or more complicated ensemble is not justified by the current results. The next meaningful test is a frozen prospective evaluation on genuinely new completed outcomes, rather than further model complexity or selecting another model after repeatedly inspecting the same years.


## Pretrained Transformer benchmark: Chronos-2 (actual zero-shot evaluation)

The separate [`chronos2_zero_shot_benchmark.ipynb`](chronos2_zero_shot_benchmark.ipynb) contains the complete executable Colab workflow. It pins the Hugging Face revision of `amazon/chronos-2`, supplies 252 past adjusted-close sessions, forecasts ten future log-price steps, converts the terminal forecast to return, and evaluates identical 2024–2026 origins.

Chronos-2 was **not trained or fine-tuned by StockSense**. Its pretrained weights remained frozen. StockSense performed inference and separately calibrated volatility-scaled intervals using the preceding fourth quarter. Calling this “Transformer training” would be inaccurate.


In [ ]:
# Recorded full-run outputs from notebooks/chronos2_zero_shot_benchmark.ipynb.
# These literals make the completed benchmark reviewable without downloading model weights.
chronos_point_results = pd.DataFrame([
    [2024, "Zero return", 21054, 4.7177, 7.1688, -0.0297, 42.2770, 50.0000, 0.0000],
    [2024, "Chronos-2",   21054, 4.7400, 7.1999, -0.0387, 54.4695, 50.8990, 0.5435],
    [2025, "Zero return", 20880, 5.1707, 7.3300, -0.0186, 43.3860, 50.0000, 0.0000],
    [2025, "Chronos-2",   20880, 5.2141, 7.3972, -0.0374, 52.2653, 49.8438, 0.4662],
    [2026, "Zero return", 14703, 5.8039, 8.4546, -0.0170, 47.3305, 50.0000, 0.0000],
    [2026, "Chronos-2",   14703, 5.9507, 8.5558, -0.0415, 50.3231, 49.4484, 0.5275],
], columns=["year", "model", "rows", "MAE_pp", "RMSE_pp", "R2",
            "direction_accuracy_pct", "balanced_accuracy_pct", "mean_prediction_pct"])
chronos_interval_results = pd.DataFrame([
    [2024, "native", 87.1046, 18.4085], [2024, "calibrated", 91.6548, 23.7776],
    [2025, "native", 87.9933, 21.3017], [2025, "calibrated", 92.9789, 27.9068],
    [2026, "native", 86.8462, 23.2868], [2026, "calibrated", 86.6966, 24.1848],
], columns=["year", "interval", "coverage_pct", "mean_width_pp"])
display(chronos_point_results)
display(chronos_interval_results)
print("Decision: retain the smaller tabular deployment; Chronos-2 did not beat zero-return error baselines.")


## Completed point-in-time SEC fundamentals comparison

The SEC experiment joined each price origin only to facts whose conservative filing-availability date was already known. It compared identical XGBoost folds with and without debt/assets, equity/assets, cash/assets, revenue/assets, net-income/assets, operating-income/assets, filing age and missingness indicators.

| Year | Model | Rounds | MAE (pp) | RMSE (pp) | Direction accuracy | Balanced accuracy |
|---|---|---:|---:|---:|---:|---:|
| 2024 | Price only | 1 | 4.6751 | 7.1200 | 57.7230% | 50.0% |
| 2024 | Price + SEC | 1 | 4.6751 | 7.1200 | 57.7230% | 50.0% |
| 2025 | Price only | 1 | 5.1178 | 7.2706 | 56.6140% | 50.0% |
| 2025 | Price + SEC | 1 | 5.1178 | 7.2706 | 56.6140% | 50.0% |
| 2026 | Price only | 1 | 5.7969 | 8.3917 | 52.6695% | 50.0% |
| 2026 | Price + SEC | 1 | 5.7966 | 8.3925 | 52.6695% | 50.0% |

The 2024/2025 predictions were unchanged because the inner-selected one-tree model did not use the additional variables materially. The 2026 MAE difference was only 0.0003 percentage points, RMSE was slightly worse, and all paired 10/20/40-session block intervals included zero. Therefore SEC fundamentals were retained for the rule-based explanation layer, where they are interpretable, rather than claimed as a demonstrated forecasting improvement.


## Dependence-aware diagnostics

Use paired loss differences aggregated by forecast-origin date, then resample contiguous date blocks while keeping all tickers on a date together. Independent row bootstrap would ignore shared market movements and overlapping 10-session outcomes. Monthly/ticker slices can expose failures concealed by averages; origin-time volatility buckets need training-defined thresholds.

The repository's `paired_intervals` implements this approach for its own `ForecastBundle`. No bootstrap runs by default here. The following optional slice cell uses only predictions already generated by the gated comparison.


In [ ]:
if predictions:
    slice_rows = []
    for (year, family), pred in predictions.items():
        for month, group in pred.groupby(pred.Date.dt.to_period("M")):
            error = group[TARGET].to_numpy()-group.prediction.to_numpy()
            slice_rows.append({"year": year, "model": family, "month": str(month), "rows": len(group),
                               "MAE_pp": np.abs(error).mean()*100,
                               "coverage": np.mean((group[TARGET]>=group.lower)&(group[TARGET]<=group.upper))})
    display(pd.DataFrame(slice_rows).head(30).round(4))
else:
    print("No fitted comparison predictions; slice evaluation skipped.")


## Optional exact repository rerun and artifact boundary

The saved repository workflow is separate from the complete family comparison above. The next cell is doubly gated, writes only under ignored `data/artifacts/forecasting/`, refuses an existing directory, and performs no downloads. It creates a fresh retrospective report/model/forecasts using the checked-in `walkforward` module. It does not update a production setting or overwrite the application's configured snapshot.

Never load untrusted joblib/pickle artifacts. A model-file hash protects against accidental changes, not executable serialization attacks. Do not commit generated model weights, forecasts, credentials or caches.


In [ ]:
if RUN_ALL_MODEL_TRAINING and SAVE_ARTIFACTS:
    from datetime import datetime, timezone
    from data.forecasting.walkforward import run
    run_name = "research-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    output = ROOT / "data/artifacts/forecasting" / run_name
    run(output, as_of=AS_OF)
else:
    print("No artifacts saved. Both execution gates must be enabled for a repository rerun.")


## Application handoff and remaining work

The authenticated app reads validated **precomputed `forecasts.csv` + `report.json`**. It serves only an explicitly supplied recommender shortlist; unavailable tickers are reported without fabricated forecasts. It does not train, unpickle models, download prices or recalibrate in HTTP requests. Stored snapshot origins are stale and are labeled accordingly.

The separate model bundle holds 27-feature preprocessing, XGBoost, parameter choices, source provenance, cutoff dates and empirical calibration. Refreshing inputs and making fresh inference is a separate operational task; this notebook does not make an old snapshot current. Recalibration after refitting requires new residuals from a model that did not train on those outcomes.

For each row we know finalized past/current price-volume indicators and market context; we try to predict the asset's adjusted-price return over its next 10 sessions. **X = 27 relative forecasting features. y = target_return_10d.** Uncertainty = empirically estimated interval width in percentage points, not accuracy or a probability the point prediction is correct.

Next evidence should be a frozen prospective procedure on genuinely new finalized data, with coverage, width, point errors and always-UP comparisons. No Transformer has been trained, no new dataset is downloaded here, and no independent superiority claim is warranted.


## Manual local deployment package
Production forecasting uses the selected saved XGBoost bundle and an uploaded 27-feature snapshot, not numeric SQL. Local refresh/retraining is explicit through `python -m data.forecasting.local_retrain` with either existing Parquet or `--download-yahoo`. See `data/FORECASTING.md` for completed-session cutoffs, validation/calibration boundaries and package files. No research cell trains in the web request, schedules a download or deploys to Azure. Between uploads, forecasts retain their snapshot origin dates.
